# Phase 5: EDA and Statistical Decision Layer

## Goals
1. Build descriptive understanding across churn, demand, campaign, returns, and store performance.
2. Quantify uncertainty with confidence intervals and statistical tests.
3. Produce decision thresholds and guardrails for downstream modeling phases.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.tsa.stattools import acf
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style='whitegrid')

CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / 'data').exists() else CWD.parent
DATA = ROOT / 'data' / 'processed'
OUT = ROOT / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)

print('Root:', ROOT)
print('Data dir exists:', DATA.exists())

Root: C:\Users\USER\Documents\Python Projects\retail-intelligence
Data dir exists: True


In [2]:
customer = pd.read_csv(DATA / 'mart_customer_features.csv', parse_dates=['signup_date','first_order_date','last_order_date'])
demand = pd.read_csv(DATA / 'mart_product_demand.csv', parse_dates=['week_start_date'])
campaign = pd.read_csv(DATA / 'mart_campaign_response.csv', parse_dates=['assignment_datetime'])
returns = pd.read_csv(DATA / 'mart_returns_risk.csv', parse_dates=['order_date'])
store = pd.read_csv(DATA / 'mart_store_week_performance.csv', parse_dates=['week_start_date'])

for name, df in [('customer', customer), ('demand', demand), ('campaign', campaign), ('returns', returns), ('store', store)]:
    print(name, df.shape)

customer (50000, 40)
demand (639673, 26)
campaign (102593, 28)
returns (1246512, 32)
store (5512, 26)


## Required Charts
- Churn distribution
- Demand trend by category
- Return rate by discount band
- Campaign response by segment
- Segment scatterplots

## Statistical Plan
- Confidence intervals: AOV, conversion rate, return rate, weekly demand mean.
- Hypothesis tests: basket value t-test (high vs low discount), conversion z-test (treatment vs control).
- Time-series diagnostics: rolling mean and autocorrelation of weekly units.
- Bayesian update: conversion posterior for treatment group with Beta-Binomial model.
- Outputs: chart PNG files and `phase5_stat_summary.csv`.

In [3]:
chart_registry = []

# 1) Churn distribution
plt.figure(figsize=(6, 4))
churn_share = customer['churn_flag_90d'].value_counts(normalize=True).sort_index()
ax = churn_share.plot(kind='bar', color=['#4CAF50', '#F44336'])
ax.set_xticklabels(['Not Churn', 'Churn'], rotation=0)
ax.set_ylabel('Share')
ax.set_title('Churn Distribution (90d)')
plt.tight_layout()
p = OUT / 'phase5_chart_churn_distribution.png'
plt.savefig(p, dpi=150)
plt.close()
chart_registry.append({'chart': 'churn_distribution', 'path': str(p)})

# 2) Demand trend by category
weekly_cat = demand.groupby(['week_start_date', 'category'], as_index=False)['units_sold'].sum()
top_cats = weekly_cat.groupby('category')['units_sold'].sum().nlargest(5).index
plot_df = weekly_cat[weekly_cat['category'].isin(top_cats)]
plt.figure(figsize=(11, 5))
sns.lineplot(data=plot_df, x='week_start_date', y='units_sold', hue='category')
plt.title('Weekly Demand Trend by Top Categories')
plt.xlabel('Week Start')
plt.ylabel('Units Sold')
plt.tight_layout()
p = OUT / 'phase5_chart_demand_trend_by_category.png'
plt.savefig(p, dpi=150)
plt.close()
chart_registry.append({'chart': 'demand_trend_by_category', 'path': str(p)})

# 3) Return rate by discount band
ret_band = returns.groupby('discount_band', as_index=False)['return_flag'].mean().sort_values('return_flag', ascending=False)
plt.figure(figsize=(7, 4))
sns.barplot(data=ret_band, x='discount_band', y='return_flag', palette='crest')
plt.title('Return Rate by Discount Band')
plt.xlabel('Discount Band')
plt.ylabel('Return Rate')
plt.tight_layout()
p = OUT / 'phase5_chart_return_rate_by_discount_band.png'
plt.savefig(p, dpi=150)
plt.close()
chart_registry.append({'chart': 'return_rate_by_discount_band', 'path': str(p)})

# 4) Campaign response by segment
segment_resp = campaign.groupby('predicted_business_segment_at_send', as_index=False)['response_flag_30d'].mean().sort_values('response_flag_30d', ascending=False)
plt.figure(figsize=(10, 4))
sns.barplot(data=segment_resp, x='predicted_business_segment_at_send', y='response_flag_30d', palette='viridis')
plt.title('Campaign 30d Response by Predicted Segment')
plt.xlabel('Predicted Segment at Send')
plt.ylabel('Response Rate')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
p = OUT / 'phase5_chart_campaign_response_by_segment.png'
plt.savefig(p, dpi=150)
plt.close()
chart_registry.append({'chart': 'campaign_response_by_segment', 'path': str(p)})

# 5) Segment scatterplot (value vs recency)
seg_scatter = customer[['customer_segment_seed', 'total_net_revenue', 'recency_days']].dropna().copy()
sample_n = min(12000, len(seg_scatter))
seg_scatter = seg_scatter.sample(n=sample_n, random_state=42)
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=seg_scatter,
    x='recency_days',
    y='total_net_revenue',
    hue='customer_segment_seed',
    alpha=0.5,
    s=20,
    linewidth=0
)
plt.title('Customer Segments: Recency vs Net Revenue')
plt.xlabel('Recency Days')
plt.ylabel('Total Net Revenue')
plt.tight_layout()
p = OUT / 'phase5_chart_customer_segment_scatter.png'
plt.savefig(p, dpi=150)
plt.close()
chart_registry.append({'chart': 'customer_segment_scatter', 'path': str(p)})

# 6) Feature importance chart (exploratory)
fi_df = customer.copy()
target = fi_df['churn_flag_90d']
candidate_cols = [
    c for c in fi_df.columns
    if c not in ['customer_id', 'signup_date', 'first_order_date', 'last_order_date', 'customer_value_band', 'churn_flag_90d']
]
X = fi_df[candidate_cols].copy()
num_cols = X.select_dtypes(include=['number', 'bool']).columns.tolist()
X_num = X[num_cols].fillna(0)
rf = RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1)
rf.fit(X_num, target)
imp = pd.DataFrame({'feature': num_cols, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False).head(15)
plt.figure(figsize=(8, 6))
sns.barplot(data=imp, x='importance', y='feature', palette='magma')
plt.title('Exploratory Feature Importance for Churn')
plt.tight_layout()
p = OUT / 'phase5_chart_feature_importance_churn.png'
plt.savefig(p, dpi=150)
plt.close()
chart_registry.append({'chart': 'feature_importance_churn', 'path': str(p)})

# 7) Forecast-vs-actual starter plot (naive baseline)
weekly_total = demand.groupby('week_start_date', as_index=False)['units_sold'].sum().sort_values('week_start_date')
weekly_total['naive_forecast'] = weekly_total['units_sold'].shift(1)
plt.figure(figsize=(11, 4))
plt.plot(weekly_total['week_start_date'], weekly_total['units_sold'], label='actual', linewidth=2)
plt.plot(weekly_total['week_start_date'], weekly_total['naive_forecast'], label='naive_forecast_lag1', linewidth=1.5, linestyle='--')
plt.title('Weekly Demand: Actual vs Naive Forecast')
plt.xlabel('Week Start')
plt.ylabel('Units Sold')
plt.legend()
plt.tight_layout()
p = OUT / 'phase5_chart_forecast_vs_actual.png'
plt.savefig(p, dpi=150)
plt.close()
chart_registry.append({'chart': 'forecast_vs_actual', 'path': str(p)})

charts_df = pd.DataFrame(chart_registry)
charts_df.to_csv(OUT / 'phase5_chart_registry.csv', index=False)
charts_df

C:\Users\USER\AppData\Local\Temp\ipykernel_29408\2896918245.py:34: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=ret_band, x='discount_band', y='return_flag', palette='crest')
C:\Users\USER\AppData\Local\Temp\ipykernel_29408\2896918245.py:47: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=segment_resp, x='predicted_business_segment_at_send', y='response_flag_30d', palette='viridis')
C:\Users\USER\AppData\Local\Temp\ipykernel_29408\2896918245.py:95: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=imp, x='importance', y='feature', 

,chart,path
0,churn_distribution,C:\Users\USER\Documents\Python Projects\retail...
1,demand_trend_by_category,C:\Users\USER\Documents\Python Projects\retail...
2,return_rate_by_discount_band,C:\Users\USER\Documents\Python Projects\retail...
3,campaign_response_by_segment,C:\Users\USER\Documents\Python Projects\retail...
4,customer_segment_scatter,C:\Users\USER\Documents\Python Projects\retail...
5,feature_importance_churn,C:\Users\USER\Documents\Python Projects\retail...
6,forecast_vs_actual,C:\Users\USER\Documents\Python Projects\retail...


In [4]:
def mean_ci(series: pd.Series, alpha: float = 0.05):
    x = series.dropna().astype(float)
    n = len(x)
    mean = x.mean() if n else np.nan
    if n <= 1:
        return mean, np.nan, np.nan
    se = x.std(ddof=1) / np.sqrt(n)
    z = stats.norm.ppf(1 - alpha / 2)
    return mean, mean - z * se, mean + z * se

results = []

# Confidence intervals
aov_mean, aov_l, aov_u = mean_ci(customer['avg_order_value'])
results.append({'metric': 'avg_order_value', 'estimate': aov_mean, 'ci_lower': aov_l, 'ci_upper': aov_u, 'test': 'mean_ci'})

conv = campaign['response_flag_30d'].astype(int)
conv_rate = conv.mean()
n_conv = conv.notna().sum()
z = stats.norm.ppf(0.975)
conv_se = np.sqrt((conv_rate * (1 - conv_rate)) / max(n_conv, 1))
results.append({'metric': 'campaign_response_rate_30d', 'estimate': conv_rate, 'ci_lower': conv_rate - z * conv_se, 'ci_upper': conv_rate + z * conv_se, 'test': 'proportion_ci'})

ret_rate = returns['return_flag'].astype(int).mean()
n_ret = returns['return_flag'].notna().sum()
ret_se = np.sqrt((ret_rate * (1 - ret_rate)) / max(n_ret, 1))
results.append({'metric': 'return_rate_item_level', 'estimate': ret_rate, 'ci_lower': ret_rate - z * ret_se, 'ci_upper': ret_rate + z * ret_se, 'test': 'proportion_ci'})

wk_mean, wk_l, wk_u = mean_ci(demand['units_sold'])
results.append({'metric': 'weekly_units_mean', 'estimate': wk_mean, 'ci_lower': wk_l, 'ci_upper': wk_u, 'test': 'mean_ci'})

# Hypothesis tests
returns_high = returns.loc[returns['discount_band'] == 'deep_discount', 'item_net_price'].dropna()
returns_low = returns.loc[returns['discount_band'] == 'low_discount', 'item_net_price'].dropna()
t_stat, t_p = stats.ttest_ind(returns_high, returns_low, equal_var=False)
results.append({'metric': 't_test_item_net_price_deep_vs_low_discount', 'estimate': t_stat, 'ci_lower': np.nan, 'ci_upper': np.nan, 'test': f'p_value={t_p:.6f}'})

tr = campaign.loc[campaign['treatment_flag'] == True, 'response_flag_30d'].astype(int)
ct = campaign.loc[campaign['control_flag'] == True, 'response_flag_30d'].astype(int)
count = np.array([tr.sum(), ct.sum()])
nobs = np.array([len(tr), len(ct)])
z_stat, z_p = proportions_ztest(count, nobs)
uplift = tr.mean() - ct.mean()
results.append({'metric': 'z_test_treatment_vs_control_response_30d', 'estimate': z_stat, 'ci_lower': np.nan, 'ci_upper': np.nan, 'test': f'p_value={z_p:.6f};uplift={uplift:.6f}'})

# Time-series diagnostics
weekly_total = demand.groupby('week_start_date', as_index=False)['units_sold'].sum().sort_values('week_start_date')
weekly_total['rolling_4w_mean'] = weekly_total['units_sold'].rolling(4).mean()
acf_vals = acf(weekly_total['units_sold'].values, nlags=8, fft=False)
results.append({'metric': 'acf_lag_1', 'estimate': float(acf_vals[1]), 'ci_lower': np.nan, 'ci_upper': np.nan, 'test': 'acf'})
results.append({'metric': 'acf_lag_4', 'estimate': float(acf_vals[4]), 'ci_lower': np.nan, 'ci_upper': np.nan, 'test': 'acf'})

# Bayesian update (Beta prior/posterior for treatment conversion)
alpha_prior, beta_prior = 1.0, 1.0
alpha_post = alpha_prior + tr.sum()
beta_post = beta_prior + (len(tr) - tr.sum())
post_mean = alpha_post / (alpha_post + beta_post)
post_l, post_u = stats.beta.ppf([0.025, 0.975], alpha_post, beta_post)
results.append({'metric': 'bayesian_treatment_conversion_posterior_mean', 'estimate': post_mean, 'ci_lower': post_l, 'ci_upper': post_u, 'test': 'beta_posterior_95_credible_interval'})

summary = pd.DataFrame(results)
summary.to_csv(OUT / 'phase5_stat_summary.csv', index=False)

# Also export a compact weekly diagnostics table
weekly_total.to_csv(OUT / 'phase5_weekly_demand_diagnostics.csv', index=False)

summary

,metric,estimate,ci_lower,ci_upper,test
0,avg_order_value,126.872386,126.243520,127.501252,mean_ci
1,campaign_response_rate_30d,0.192820,0.190406,0.195234,proportion_ci
2,return_rate_item_level,0.151141,0.150512,0.151770,proportion_ci
3,weekly_units_mean,2.239247,2.235068,2.243427,mean_ci
4,t_test_item_net_price_deep_vs_low_discount,-133.382696,NaN,NaN,p_value=0.000000
5,z_test_treatment_vs_control_response_30d,12.727821,NaN,NaN,p_value=0.000000;uplift=0.042382
6,acf_lag_1,0.667194,NaN,NaN,acf
7,acf_lag_4,0.646722,NaN,NaN,acf
8,bayesian_treatment_conversion_posterior_mean,0.199760,0.197092,0.202442,beta_posterior_95_credible_interval
